# Day 12 - Graph Traversal: BFS and DFS

On [day 11](https://medium.com/100-days-of-python/day-11-data-structure-graph-a4229c3dabaf)
we built a graph. A graph you cannot walk through is just a pile of edges, so today we
learn the two ways to walk it:

- **BFS (breadth-first search)** - explore the graph in rings around the start vertex,
  using a **queue** (day 02).
- **DFS (depth-first search)** - dive as deep as possible before backing up,
  using a **stack** (day 02) or the call stack.

Everything else in graph theory - topological sort, shortest paths, connected
components, cycle detection - is one of these two traversals with extra bookkeeping.

In [1]:
from collections import defaultdict, deque


class Graph:
    """Undirected graph stored as an adjacency list."""

    def __init__(self):
        self.adj = defaultdict(list)

    def add_edge(self, u, v):
        self.adj[u].append(v)
        self.adj[v].append(u)      # drop this line for a directed graph

    def neighbors(self, u):
        return self.adj[u]

    def vertices(self):
        return sorted(self.adj)

    def __repr__(self):
        return '\n'.join(f'{v}: {sorted(self.adj[v])}' for v in self.vertices())


g = Graph()
for u, v in [(1, 2), (1, 3), (2, 4), (2, 5), (3, 5), (3, 6),
             (4, 7), (5, 7), (6, 8), (7, 8)]:
    g.add_edge(u, v)

print(g)

1: [2, 3]
2: [1, 4, 5]
3: [1, 5, 6]
4: [2, 7]
5: [2, 3, 7]
6: [3, 8]
7: [4, 5, 8]
8: [6, 7]


## Breadth-first search

BFS keeps a **queue** of vertices it has discovered but not yet expanded. Pop one,
push all of its unseen neighbours, repeat. Because the queue is FIFO, every vertex at
distance *d* leaves the queue before any vertex at distance *d+1* - the traversal
sweeps the graph layer by layer.

The one rule that matters: **mark a vertex as visited when you push it, not when you
pop it.** Mark on pop and a vertex with two parents gets queued twice.

In [2]:
def bfs(graph, start):
    """Return (visit order, distance from start, parent pointers)."""
    visited = {start}
    dist = {start: 0}
    parent = {start: None}
    order = []
    queue = deque([start])

    while queue:
        u = queue.popleft()
        order.append(u)
        for v in sorted(graph.neighbors(u)):
            if v not in visited:
                visited.add(v)              # mark on push, not on pop
                dist[v] = dist[u] + 1
                parent[v] = u
                queue.append(v)
    return order, dist, parent


order, dist, parent = bfs(g, 1)
print('order   :', order)
print('distance:', dist)

order   : [1, 2, 3, 4, 5, 6, 7, 8]
distance: {1: 0, 2: 1, 3: 1, 4: 2, 5: 2, 6: 2, 7: 3, 8: 3}


Because BFS discovers vertices in non-decreasing distance order, `dist` is the
**shortest path length in edges** from the start to every reachable vertex. Follow
`parent` backwards and you get the path itself - this is the whole reason BFS shows up
in maze solvers, word ladders and social-network "degrees of separation".

In [3]:
def shortest_path(graph, start, goal):
    _, dist, parent = bfs(graph, start)
    if goal not in parent:
        return None                     # unreachable
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]


print('1 -> 8 :', shortest_path(g, 1, 8), '  length =', dist[8])

1 -> 8 : [1, 3, 6, 8]   length = 3


## Depth-first search

DFS is the mirror image: swap the queue for a stack. The recursive version is the one
worth memorising because it reads like the definition.

DFS splits the edges into **tree edges** (the ones it walks) and everything else. Which
edges end up in which group is the raw material for cycle detection, bridges,
articulation points and strongly connected components later in the series.

In [4]:
def dfs_recursive(graph, start, visited=None, order=None):
    if visited is None:
        visited, order = set(), []
    visited.add(start)
    order.append(start)
    for v in sorted(graph.neighbors(start)):
        if v not in visited:
            dfs_recursive(graph, v, visited, order)
    return order


def dfs_iterative(graph, start):
    """Same traversal without recursion - useful when the graph is deep."""
    visited, order = set(), []
    stack = [start]
    while stack:
        u = stack.pop()
        if u in visited:
            continue
        visited.add(u)
        order.append(u)
        # reversed() so the smallest neighbour is expanded first
        for v in sorted(graph.neighbors(u), reverse=True):
            if v not in visited:
                stack.append(v)
    return order


print('recursive:', dfs_recursive(g, 1))
print('iterative:', dfs_iterative(g, 1))

recursive: [1, 2, 4, 7, 5, 3, 6, 8]
iterative: [1, 2, 4, 7, 5, 3, 6, 8]


Note the two versions do **not** have to agree in general - the iterative one
re-pushes vertices that may already be on the stack, so it can pop them in a different
order. Both are valid depth-first traversals.

A graph is not always connected, so a full traversal loops over every vertex:

In [5]:
def connected_components(graph):
    seen, comps = set(), []
    for v in graph.vertices():
        if v not in seen:
            comp = dfs_recursive(graph, v)
            seen |= set(comp)
            comps.append(comp)
    return comps


h = Graph()
for u, v in [(1, 2), (2, 3), (10, 11), (20, 21), (21, 22)]:
    h.add_edge(u, v)
print(connected_components(h))

[[1, 2, 3], [10, 11], [20, 21, 22]]


## BFS or DFS?

| | BFS | DFS |
|---|---|---|
| container | queue | stack / recursion |
| finds | shortest path in an unweighted graph | *a* path, any path |
| memory | O(width) - can be huge | O(depth) |
| natural for | levels, distances, "closest" | cycles, components, ordering, backtracking |

Both run in **O(V + E)** time: every vertex is pushed once and every edge is inspected
once (twice in an undirected graph, once from each side).

## LeetCode 200 - Number of Islands

> Given an `m x n` binary grid where `'1'` is land and `'0'` is water, return the number
> of islands. An island is surrounded by water and is formed by connecting adjacent
> lands horizontally or vertically.

The trick is to notice this **is** a graph problem in disguise: every land cell is a
vertex, and two land cells share an edge when they are orthogonally adjacent. Counting
islands is counting connected components - exactly what we just wrote.

In [6]:
def num_islands_bfs(grid):
    if not grid or not grid[0]:
        return 0
    rows, cols = len(grid), len(grid[0])
    grid = [list(row) for row in grid]      # work on a mutable copy
    count = 0

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != '1':
                continue
            count += 1                      # found a fresh island
            queue = deque([(r, c)])
            grid[r][c] = '0'                # sink it: the grid is our visited set
            while queue:
                i, j = queue.popleft()
                for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    ni, nj = i + di, j + dj
                    if 0 <= ni < rows and 0 <= nj < cols and grid[ni][nj] == '1':
                        grid[ni][nj] = '0'
                        queue.append((ni, nj))
    return count


def num_islands_dfs(grid):
    """Same idea, recursive flood fill. Watch the recursion depth on big grids."""
    if not grid or not grid[0]:
        return 0
    rows, cols = len(grid), len(grid[0])
    grid = [list(row) for row in grid]

    def sink(i, j):
        if not (0 <= i < rows and 0 <= j < cols) or grid[i][j] != '1':
            return
        grid[i][j] = '0'
        sink(i + 1, j); sink(i - 1, j); sink(i, j + 1); sink(i, j - 1)

    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                count += 1
                sink(r, c)
    return count

In [7]:
tests = [
    (['11110', '11010', '11000', '00000'], 1),
    (['11000', '11000', '00100', '00011'], 3),
    ([], 0),
    (['000'], 0),
]

for grid, expected in tests:
    got_bfs, got_dfs = num_islands_bfs(grid), num_islands_dfs(grid)
    assert got_bfs == got_dfs == expected, (grid, got_bfs, got_dfs, expected)
    print(f'{str(grid):50s} -> {got_bfs}  ok')

print('\nall tests passed')

['11110', '11010', '11000', '00000']               -> 1  ok
['11000', '11000', '00100', '00011']               -> 3  ok
[]                                                 -> 0  ok
['000']                                            -> 0  ok

all tests passed


Time complexity is **O(m x n)**: every cell is looked at a constant number of
times and sunk at most once. Space is **O(min(m, n))** for the BFS queue in the worst
case, or **O(m x n)** for the DFS call stack - which is why the iterative version is the
safer answer in an interview.

Sinking the land instead of keeping a separate `visited` set is the small trick that
makes this solution short. If the input must not be mutated, copy it first (as above)
or keep a `set` of visited coordinates.

## What this unlocks

BFS and DFS are the engine under most of the next week: **day 13 topological sort** is a
DFS with a post-order stamp (or a BFS on in-degrees), **day 15/16 minimum spanning
trees** grow a tree the same way, and **day 17 Dijkstra** is BFS with the queue swapped
for a heap - the day 05 priority queue finally earning its keep.

Tomorrow: topological sort, and why a course schedule is really a DAG.